In [ ]:
from ollama import chat

response = chat(
    model="qwen3-vl:8b",
    messages=[
        {
            "role": "user",
            "content": "Describe this research paper figure in detail. Explain every graph and axis.",
            "images": ["/home/jugal/Documents/DataScience/Advanced_RAG/Research_Paper_rag/img7.jpg"]   # Local image path
        }
    ]
)

print(response["message"]["content"])

### Detailed Description of the Research Paper Figure  

This figure is a **scatter plot with an overlaid trend line**, used to visualize the relationship between two quantitative variables: **GER** (x-axis) and **NER** (y-axis). Below is a breakdown of its components and structure:  


---

#### **1. Axes and Scales**  
- **X-axis (Horizontal)**: Labeled *GER*, with a range from **0 to 150**.  
  - *Interpretation*: GER is the independent variable (e.g., a predictor, input, or experimental condition). The scale spans from 0 (no value) to 150 (maximum observed value).  
- **Y-axis (Vertical)**: Labeled *NER*, with a range from **0 to 100**.  
  - *Interpretation*: NER is the dependent variable (e.g., an outcome, response, or measured result). The scale spans from 0 (no value) to 100 (maximum observed value).  

The axes are linear and use consistent intervals, suggesting a quantitative comparison between the two variables.  


---

#### **2. Data Representation**  
- **Scatter Points**

In [6]:
import os
import base64
import torch
from dotenv import load_dotenv
from PIL import Image

from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, AutoModel

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage

# ==========================================
# 1. Initialization & Setup
# ==========================================
load_dotenv("/home/jugal/Documents/DataScience/Advanced_RAG/.env")
if "NVIDIA_API_KEY" not in os.environ:
    raise ValueError("⚠️ NVIDIA_API_KEY not found in .env file!")

# Initialize Qdrant Client
qdrant_client = QdrantClient(host="localhost", port=6333)
TEXT_COLLECTION = "research_papers_text_v5"
IMAGE_COLLECTION = "research_papers_image_v6"

print("Loading models... This may take a minute.")

# Text Embedding Models
dense_text_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")
sparse_text_model = SparseTextEmbedding(model_name="Qdrant/bm25")

# Vision LLM (NVIDIA NIM)
vision_llm = ChatNVIDIA(model="nvidia/llama-3.1-nemotron-nano-vl-8b-v1")

# SigLIP Custom Wrapper (For Image Vectorstore Retrieval)
class SiglipEmbedder:
    def __init__(self, model_name="google/siglip-base-patch16-512", device="cpu"):
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.device = device
        
    def encode(self, text):
        if isinstance(text, str):
            text = [text]
        inputs = self.processor(text=text, padding="max_length", return_tensors="pt").to(self.device)
        with torch.no_grad():
            text_features = self.model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        if len(text) == 1:
            return text_features[0].cpu().numpy()
        return text_features.cpu().numpy()

siglip_model = SiglipEmbedder(device="cpu")

# ==========================================
# 2. Retrieval Helper Functions
# ==========================================
def compute_weighted_fusion(list1, list2, alpha=0.4):
    scores_1 = {p.id: p.score for p in list1}
    scores_2 = {p.id: p.score for p in list2}
    
    def normalize(scores):
        if not scores: return scores
        min_val, max_val = min(scores.values()), max(scores.values())
        if max_val == min_val: return {k: 1.0 for k in scores}
        return {k: (v - min_val) / (max_val - min_val) for k, v in scores.items()}
        
    norm_1, norm_2 = normalize(scores_1), normalize(scores_2)
    fused_scores, point_map = {}, {}
    
    for p in list1:
        point_map[p.id] = p
        fused_scores[p.id] = alpha * norm_1.get(p.id, 0)
    for p in list2:
        point_map[p.id] = p
        fused_scores[p.id] = fused_scores.get(p.id, 0) + (1 - alpha) * norm_2.get(p.id, 0)
            
    sorted_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    return [point_map[pid] for pid in sorted_ids]

def retrieve_top_text_chunks(query, top_k=3):
    # mxbai-embed-large-v1 requires this exact prompt for queries to work properly!
    query_prefix = "Represent this sentence for searching relevant passages: "
    dense_vec = dense_text_model.encode(query_prefix + query)
    
    sparse_emb = list(sparse_text_model.embed([query]))[0]
    sparse_vec = models.SparseVector(
        indices=sparse_emb.indices.tolist(),
        values=sparse_emb.values.tolist()
    )

    dense_results = qdrant_client.query_points(
        collection_name=TEXT_COLLECTION,
        query=dense_vec.tolist(),
        using="dense", limit=10, with_payload=True
    ).points
    
    sparse_results = qdrant_client.query_points(
        collection_name=TEXT_COLLECTION,
        query=sparse_vec,
        using="sparse", limit=10, with_payload=True
    ).points
    
    fused_results = compute_weighted_fusion(dense_results, sparse_results, alpha=0.4)
    return [point.payload.get("text", "") for point in fused_results[:top_k]]


def retrieve_top_images(query, top_k=2):
    # Encode the text query into the SigLIP space
    query_vector = siglip_model.encode(query)
    
    results = qdrant_client.query_points(
        collection_name=IMAGE_COLLECTION,
        query=query_vector.tolist(),
        limit=top_k, 
        with_payload=True
    ).points
    
    # Extract the image paths from the payload
    image_paths = []
    for point in results:
        path = point.payload.get("image_path")
        if path and os.path.exists(path):
            image_paths.append(path)
    return image_paths

# ==========================================
# 3. Generation Function
# ==========================================
def encode_image_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def generate_answer(query, text_chunks, image_paths):
    # 1. Format the retrieved context
    context_str = "\n\n---\n\n".join([f"Chunk {i+1}:\n{text}" for i, text in enumerate(text_chunks)])
    
    system_prompt = (
        "You are an expert AI research assistant. You have been provided with excerpts from a research paper "
        "and relevant figures/images extracted from the document.\n\n"
        "Use the provided Text Context and the provided Images to answer the user's question accurately.\n"
        f"TEXT CONTEXT:\n{context_str}"
    )

    # 2. Build the multimodal message content
    message_content = [
        {"type": "text", "text": f"{system_prompt}\n\nUSER QUESTION: {query}"}
    ]
    
    # Append the retrieved images to the prompt
    for img_path in image_paths:
        b64_img = encode_image_base64(img_path)
        img_ext = img_path.split('.')[-1].lower()
        mime_type = f"image/{img_ext}" if img_ext in ['jpeg', 'png'] else 'image/jpeg'
        
        message_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:{mime_type};base64,{b64_img}"}
        })

    message = HumanMessage(content=message_content)
    
    # 3. Call the Vision LLM
    print("\nGenerating answer with Nemotron Nano VL...")
    response = vision_llm.invoke([message])
    return response.content
# ==========================================
# 4. Pipeline Execution
# ==========================================
def multimodal_rag_pipeline(query):
    print(f"\n[QUERY] {query}")
    print("=" * 60)
    
    # Step 1: Retrieve
    print("🔍 Retrieving top 3 text chunks...")
    top_texts = retrieve_top_text_chunks(query, top_k=3)
    
    for i, text in enumerate(top_texts):
        # Print a short preview of each chunk to verify
        print(f"  ➜ Text Chunk {i+1} Preview: {text[:150]}...")
    
    print("\n🖼️ Retrieving top 2 images...")
    top_images = retrieve_top_images(query, top_k=2)
    for i, img_path in enumerate(top_images):
        print(f"  ➜ Image {i+1} Path: {img_path}")
    
    print("=" * 60)
    
    # Step 2: Generate
    answer = generate_answer(query, top_texts, top_images)
    
    print("\n===============================")
    print("🤖 LLM ANSWER:")
    print("===============================")
    print(answer)
    
    # Return all the data so you can inspect it in your notebook
    return {
        "query": query,
        "answer": answer,
        "retrieved_texts": top_texts,
        "retrieved_images": top_images
    }

# --- RUN A TEST ---
if __name__ == "__main__":
    test_query = test_query = "How can we use multiple imputation for hierarchical nonlinear time series data?"
    result = multimodal_rag_pipeline(test_query)
    # You can now access result["retrieved_texts"] or result["retrieved_images"] directly!

Loading models... This may take a minute.

[QUERY] How can we use multiple imputation for hierarchical nonlinear time series data?
🔍 Retrieving top 3 text chunks...
  ➜ Text Chunk 1 Preview: Paper Title: Multiple Imputation of Hierarchical Nonlinear Time Series Data with an
  Application to School Enrollment Data
Section Title: #### Abstra...
  ➜ Text Chunk 2 Preview: Paper Title: Multiple Imputation of Hierarchical Nonlinear Time Series Data with an
  Application to School Enrollment Data
Section Title: #### Abstra...
  ➜ Text Chunk 3 Preview: Paper Title: Multiple Imputation of Hierarchical Nonlinear Time Series Data with an
  Application to School Enrollment Data
Section Title: # 5.2. Anal...

🖼️ Retrieving top 2 images...
  ➜ Image 1 Path: /home/jugal/Documents/DataScience/Advanced_RAG/Research_Paper_rag/precessed_data/images/2403.04635v2_img-11.jpeg
  ➜ Image 2 Path: /home/jugal/Documents/DataScience/Advanced_RAG/Research_Paper_rag/precessed_data/images/2403.05373v2_img-6.jpeg

Ge

In [7]:
result

{'query': 'How can we use multiple imputation for hierarchical nonlinear time series data?',
 'answer': "Multiple imputation is a statistical technique used to handle missing data. In the context of hierarchical nonlinear time series data, like the school enrollment rates data you mentioned, multiple imputation can be particularly useful when the data has a non-linear relationship between variables. This method creates multiple versions of the data where the missing values are replaced by simulated values, based on the information provided by the non-missing data and any auxiliary variables.\n\nTo use multiple imputation for hierarchical nonlinear time series data:\n\n1. Choose an appropriate imputation model: This model should accurately capture the non-linear relationship between the variables of interest.\n2. Select an imputation method: There are various methods available, such as the Multivariate Imputation by Chained Equations (MICE) or Bayesian methods.\n3. Specify the number of

## Building the Inference Pipeline:

- WHen the User queries the application
- The Request gets routed if it needed the RAG or it can ans directly
- Then the ans 
- The memory is maintained of simple context memory 
- Logging will be happend 
- Uses LangGraph to manage the state of the memory
- Make a Different Repo